In [22]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [23]:
import sys
from pathlib import Path

def find_repo_root(start: Path, marker=".git") -> Path:
    for parent in [start, *start.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find repo root from {start}")

repo_base = find_repo_root(Path.cwd())
# repo_base
tools_path = repo_base / "tools"
sys.path.append(str(tools_path))

In [24]:
from battle import *

In [30]:
log_dir.relative_to(repo_base)

PosixPath('replays/gen9-randombattle')

In [ ]:
log_dir = repo_base / f"replays/gen9-randombattle/"
logs = Path('./../../data/replays/gen9-randombattle/')

for i in range(5):
    print(next(log_dir.iterdir()))

In [37]:
rows = []

stat_names = ["hp","atk","def","spa","spd","spe"]

for file in logs.iterdir():
    if file.is_file():
        battle = Battle(logs/file.name)
        if not battle.custom_ruleQ:
            to_add = { # initial non-team data we may want to train on
                "id": battle.id,
                "rated": battle.rated,
                "duration": battle.end_time - battle.start_time,
                "player1": battle.player1.name,
                "player2": battle.player2.name,
                "p1_rating" : battle.player1.elo0,
                "p2_rating": battle.player2.elo0,
                "p1_wins" : battle.player1.name == battle.winner.name,
            }

            # team data and stats for each mon
            for player_index in range(2):
                for mon_index, mon_name in enumerate(battle.teams_full[player_index].keys()):
                    to_add["p" + str(player_index + 1) + "_mon_" + str(mon_index + 1) + "_species_id"] = battle.teams_full[player_index][mon_name]["speciesId"]
                    for stat in stat_names:
                        to_add["p" + str(player_index + 1) + "_mon_" + str(mon_index + 1) + "_" + stat] = battle.teams_full[player_index][mon_name]["stats"][stat]
            rows.append(to_add)

AttributeError: 'Battle' object has no attribute 'p1'

In [12]:
matches = pd.DataFrame(rows)

for player_index in range(2):
    for mon_index in range(6):
        new_col_name = "p" + str(player_index + 1) + "_mon_" + str(mon_index + 1) + "_attacking_stat"
        old_col_names = ["p" + str(player_index + 1) + "_mon_" + str(mon_index + 1) + "_atk",
                        "p" + str(player_index + 1) + "_mon_" + str(mon_index + 1) + "_spa"]
        matches[new_col_name] = matches[old_col_names].max(axis=1)
        matches = matches.drop(columns=old_col_names)